# HSE RecSys Challenge 2026

Нужно рекомендовать каждому пользователю 10 фильмов. Метрика — **recall@10**.

Итоговая модель сочетает глобальную EASE, локальную EASE и направленные ridge-модели
с разными весами недавней истории. Командный вариант получил 0,2447 на скрытом лидерборде,
а этот улучшенный вариант — **0,2480** по сообщению команды.

Запустите все ячейки один раз: ноутбук пересчитает сравнение на пяти временных срезах,
обучит финальную модель и сохранит `submission_improved.csv`. Четыре входных CSV
должны лежать в папке `data/`. Если в `models/` есть снепшоты, финальные матрицы загружаются из них. Другие скрипты для запуска не нужны.


In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'

import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import scipy.linalg as la
from IPython.display import display

DATA = Path('data')
MODELS = Path('models')
USE_SNAPSHOTS = True   # True: если снепшоты есть, финальные матрицы загружаются, а не обучаются заново
K = 10
PARAMS = dict(global_lam=700, local_lam=100, window=10,
              ridge_lam=100, tau=3, horizon=12, beta=64,
              tau_long=6, recent_strength=3, recent_window=5,
              recent_mix=0.25, long_mix=0.5, local_beta=4)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.precision', 5)

events = pd.read_csv(DATA / 'events.csv')
users = pd.read_csv(DATA / 'user_features.csv').sort_values('user_id').reset_index(drop=True)
items = pd.read_csv(DATA / 'item_features.csv').sort_values('item_id').reset_index(drop=True)
sample = pd.read_csv(DATA / 'submission_sample.csv')
N_USERS, N_ITEMS = len(users), len(items)
print(f'numpy={np.__version__}, pandas={pd.__version__}, scipy={scipy.__version__}')


numpy=2.5.3, pandas=3.0.5, scipy=1.18.1


## 1. Что есть в данных

У каждого пользователя есть история. Повторных пар «пользователь — фильм» нет, поэтому уже оценённые
фильмы исключаем из рекомендаций. Матрицу взаимодействий делаем бинарной: низкая оценка тоже означает,
что пользователь выбрал этот фильм.

По пояснению организатора timestamp сохраняет порядок событий. Это не календарное время.
Пропуски в нумерации сами по себе не показывают, какие фильмы попадут в тест.

In [2]:
assert not events.isna().any().any()
assert not events.duplicated(['user_id', 'item_id']).any()
assert not events.duplicated(['user_id', 'timestamp']).any()
assert np.array_equal(users.user_id.values, np.arange(N_USERS))
assert np.array_equal(items.item_id.values, np.arange(N_ITEMS))
assert set(events.user_id) == set(users.user_id) == set(sample.user_id)
assert set(events.item_id) <= set(items.item_id)

history = events.groupby('user_id').timestamp.agg(['min', 'max', 'size'])
display(pd.Series({
    'Оценок': len(events), 'Пользователей': N_USERS, 'Фильмов в признаках': N_ITEMS,
    'Фильмов с оценками': events.item_id.nunique(),
    'Минимальная история': history['size'].min(),
    'Медианная история': history['size'].median(),
    'Максимальная история': history['size'].max(),
    'Пропусков внутри нумерации': (history['max'] + 1 - history['size']).sum(),
}, name='Значение').to_frame())
display(events.rating.value_counts(normalize=True).sort_index().rename('Доля оценок').to_frame())


,Значение
Оценок,894149.0
Пользователей,6040.0
Фильмов в признаках,3706.0
Фильмов с оценками,3690.0
Минимальная история,14.0
Медианная история,84.5
Максимальная история,2056.0
Пропусков внутри нумерации,98628.0


,Доля оценок
rating,
1,0.05621
2,0.10737
3,0.26150
4,0.34913
5,0.22580


## 2. Гипотезы и валидация

| Гипотеза | Проверка |
|---|---|
| Общие предпочтения важнее одной популярности | Популярное против EASE |
| Последние просмотры описывают ближайший интерес | Соседи пропусков против недавней истории |
| Направление последовательности полезно | Локальная EASE против предсказания следующего фильма |
| Общий вкус дополняет недавний контекст | Последовательная модель отдельно и в смеси с EASE |
| Способ смешивания влияет на результат | Сумма скоров против объединения рангов |
| Дополнительные масштабы истории и последние обучающие события полезны | Финальный ансамбль против командного варианта |

Точная схема скрытого теста не опубликована. Используем пять временных срезов: скрываем один
из последних пяти известных фильмов каждого пользователя и предсказываем его по прошлому.
Все события после цели тоже удаляем. В каждом срезе одна цель на пользователя, поэтому
recall@10 равен доле попаданий в десятку. Timestamp не перенумеровываем; все матрицы
обучаем заново только на разрешённой истории. Срезы пересекаются и использовались для подбора,
поэтому они не являются независимым подтверждением качества на скрытом тесте.


In [3]:
def to_matrix(df, values=None):
    values = np.ones(len(df), np.float32) if values is None else np.asarray(values, np.float32)
    return sp.csr_matrix((values, (df.user_id, df.item_id)), shape=(N_USERS, N_ITEMS))


def temporal_split(df, cut=1):
    d = df.sort_values(['user_id', 'timestamp'])
    from_end = d.groupby('user_id').cumcount(ascending=False)
    train = d[from_end >= cut].reset_index(drop=True)
    target = d[from_end == cut - 1].reset_index(drop=True)
    assert len(target) == N_USERS
    assert (train.groupby('user_id').timestamp.max()
            < target.set_index('user_id').timestamp).all()
    return train, target


def topk(scores, X, k=K):
    scores = np.array(scores, dtype=np.float32, copy=True)
    scores[X.nonzero()] = -np.inf
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.take_along_axis(scores, top, 1).argsort(1)[:, ::-1]
    return np.take_along_axis(top, order, 1)


def recall_per_user(rec, target):
    truth = to_matrix(target).toarray().astype(bool)
    hits = truth[np.arange(N_USERS)[:, None], rec].sum(1)
    return hits / truth.sum(1)


## 3. Общие предпочтения: EASE

EASE учится по совместным оценкам фильмов. Для бинарной матрицы $X$:
$P=(X^\top X+\lambda I)^{-1}$, $B_{ij}=-P_{ij}/P_{jj}$, $B_{ii}=0$.
Скор пользователя — $XB$. Используем $\lambda=700$.
Это персонализированный ориентир для сравнения с популярностью.

In [4]:
def ease_B(X, lam=700):
    gram = (X.T @ X).toarray().astype(np.float64)
    gram[np.diag_indices(X.shape[1])] += lam
    precision = np.linalg.inv(gram)
    B = -precision / np.diag(precision)
    np.fill_diagonal(B, 0)
    return B.astype(np.float32)


## 4. Ближайший контекст: локальная EASE

Обучаем EASE на окнах из 10 позиций. Усредняем статистику окон со сдвигами 0 и 5,
чтобы близкие фильмы не разделялись произвольной границей окна.

Сравниваем два контекста: соседи пропусков в нумерации и последние просмотры.
Во втором случае вес события — $\exp(-d/\tau)$, где $d$ — расстояние от последней позиции,
$\tau=3$. Это расстояние в порядке оценок, а не в днях.

In [5]:
def window_matrix(df, length=10, offset=0):
    d = df.sort_values(['user_id', 'timestamp'])
    key = pd.factorize(d.user_id.values.astype(np.int64) * 100_000
                       + (d.timestamp.values + offset) // length)[0]
    return sp.csr_matrix((np.ones(len(d), np.float32), (key, d.item_id.values)),
                         shape=(key.max() + 1, N_ITEMS))


def local_ease_B(df, length=10, lam=100):
    windows = [window_matrix(df, length, offset) for offset in (0, length // 2)]
    gram = sum((w.T @ w).toarray().astype(np.float64) for w in windows) / 2
    gram[np.diag_indices(N_ITEMS)] += lam
    precision = np.linalg.inv(gram)
    B = -precision / np.diag(precision)
    np.fill_diagonal(B, 0)
    return B.astype(np.float32)


def tail_weights(df, tau=3, horizon=None):
    distance = (df.groupby('user_id').timestamp.transform('max') - df.timestamp).to_numpy()
    weights = np.exp(-distance / tau)
    if horizon is not None:
        weights *= distance < horizon
    return to_matrix(df, weights)


def gap_weights(df, tau=2, tail_weight=0.5):
    rows, cols, values = [], [], []
    for u, g in df.sort_values('timestamp').groupby('user_id'):
        pos = g.timestamp.values
        present = np.zeros(pos.max() + 2, bool)
        present[pos] = True
        gaps = np.flatnonzero(~present)
        weights = np.exp(-(np.abs(pos[:, None] - gaps[None, :]) - 1) / tau)
        weights[:, -1] *= tail_weight
        rows.append(np.full(len(pos), u))
        cols.append(g.item_id.values)
        values.append(weights.sum(1))
    return sp.csr_matrix((np.concatenate(values).astype(np.float32),
                          (np.concatenate(rows), np.concatenate(cols))),
                         shape=(N_USERS, N_ITEMS))


## 5. Предсказание следующего фильма

Для каждой известной оценки берём предыдущие события того же пользователя в пределах
12 позиций. Вес события на расстоянии $d$ равен $\exp(-(d-1)/\tau)$.
Матрица $C$ содержит такие контексты, а $Y$ — единицу для целевого фильма.

Обычная ridge-модель решает $B=(C^\top C+100I)^{-1}C^\top Y$. Диагональ обнуляем.
Дополнительный вариант повышает вес обучающих примеров около конца разрешённой истории:
$w=1+3\exp(-a/5)$, где $a$ — расстояние от последней доступной позиции пользователя.
Для него решаем $B=(C^\top W C+100I)^{-1}C^\top W Y$.
В финальной смеси используются обе модели с $\tau=3$ и обычная модель с $\tau=6$.


In [6]:
def directed_ridge(df, tau=3, lam=100, horizon=12, recent_strength=0, recent_window=5):
    d = df.sort_values(['user_id', 'timestamp'])
    user, item, pos = d.user_id.to_numpy(), d.item_id.to_numpy(), d.timestamp.to_numpy()
    n = len(d)
    rows, cols, values = [], [], []
    for offset in range(1, horizon + 1):
        previous = np.arange(n - offset)
        target = previous + offset
        same_context = ((user[previous] == user[target])
                        & ((pos[target] - pos[previous]) <= horizon))
        previous, target = previous[same_context], target[same_context]
        rows.append(target)
        cols.append(item[previous])
        values.append(np.exp(-((pos[target] - pos[previous]) - 1) / tau).astype(np.float32))
    C = sp.csr_matrix((np.concatenate(values), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(n, N_ITEMS))
    Y = sp.csr_matrix((np.ones(n, np.float32), (np.arange(n), item)), shape=(n, N_ITEMS))
    if recent_strength:
        age = d.groupby('user_id').timestamp.transform('max').to_numpy() - pos
        sqrt_weights = np.sqrt((1 + recent_strength * np.exp(-age / recent_window)).astype(np.float32))
        C = C.multiply(sqrt_weights[:, None]).tocsr()
        Y = Y.multiply(sqrt_weights[:, None]).tocsr()
    gram = (C.T @ C).toarray().astype(np.float64)
    cross = (C.T @ Y).toarray().astype(np.float64)
    gram[np.diag_indices(N_ITEMS)] += lam
    B = la.cho_solve(la.cho_factor(gram, lower=True, check_finite=False),
                    cross, check_finite=False).astype(np.float32)
    np.fill_diagonal(B, 0)
    return B


def improved_score(df, global_score, local_score, next_score, B_recent=None, B_long=None):
    """Итоговый ансамбль; все модели учатся только на переданной истории."""
    tail_short = tail_weights(df, PARAMS['tau'], PARAMS['horizon'])
    if B_recent is None:
        B_recent = directed_ridge(df, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon'],
                                  PARAMS['recent_strength'], PARAMS['recent_window'])
    recent_score = np.asarray(tail_short @ B_recent)
    if B_long is None:
        B_long = directed_ridge(df, PARAMS['tau_long'], PARAMS['ridge_lam'], PARAMS['horizon'])
    long_score = np.asarray(tail_weights(df, PARAMS['tau_long'], PARAMS['horizon']) @ B_long)
    directed = (1 - PARAMS['recent_mix']) * next_score + PARAMS['recent_mix'] * recent_score
    directed = (1 - PARAMS['long_mix']) * directed + PARAMS['long_mix'] * long_score
    result = global_score + PARAMS['beta'] * directed
    return result + PARAMS['local_beta'] * local_score


## 6. Сравнение моделей

Командный вариант: **global EASE + 64 × directed ridge**. Финальный вариант добавляет
локальную EASE, вторую направленную модель с более длинным контекстом и ridge с повышенным
весом недавних обучающих примеров:

```text
short = 0.75 * directed_tau3 + 0.25 * directed_recent_tau3
score = global_EASE + 64 * (0.5 * short + 0.5 * directed_tau6) + 4 * local_EASE
```

Для сравнения также оставлены популярность, соседи пропусков, локальная EASE и RRF.
В RRF используются топ-200 и слагаемое $1/(20+\text{rank})$. Параметры зафиксированы
в `PARAMS`; пять временных срезов полностью переобучаются при каждом запуске.


In [7]:
def rrf(first, second, X):
    scores = np.zeros_like(first)
    for source in (first, second):
        rec = topk(source, X, k=200)
        np.add.at(scores, (np.arange(N_USERS)[:, None], rec),
                  1 / (20 + np.arange(1, 201)[None, :]))
    return scores


MODEL_NAMES = {
    'popularity': 'Популярное',
    'global_EASE': 'EASE',
    'current_gap_blend': 'EASE + соседи пропусков',
    'tail_local_tau3_beta16': 'EASE + недавняя локальная EASE',
    'directed_ridge': 'Directed ridge',
    'rrf_local_directed': 'RRF двух последовательных моделей',
    'global_plus_directed': 'EASE + directed ridge',
    'final_candidate': 'Финальный ансамбль',
}


In [8]:
results, per_user = [], {}
started = time.time()
for cut in (1, 2, 3, 4, 5):
    train, target = temporal_split(events, cut)
    X = to_matrix(train)
    global_score = np.asarray(X @ ease_B(X, PARAMS['global_lam']))
    B_local = local_ease_B(train, PARAMS['window'], PARAMS['local_lam'])
    local_score = np.asarray(tail_weights(train, PARAMS['tau']) @ B_local)
    B_next = directed_ridge(train, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon'])
    next_score = np.asarray(tail_weights(train, PARAMS['tau'], PARAMS['horizon']) @ B_next)

    def evaluate(name, scores):
        values = recall_per_user(topk(scores, X), target)
        per_user[(cut, name)] = values
        results.append({'cut': cut, 'model': name, 'recall10': values.mean()})

    pop = np.asarray(X.sum(0)).ravel()
    evaluate('popularity', np.tile(pop, (N_USERS, 1)))
    evaluate('global_EASE', global_score)
    evaluate('current_gap_blend', global_score + 2 * np.asarray(gap_weights(train) @ B_local))
    evaluate('tail_local_tau3_beta16', global_score + 16 * local_score)
    evaluate('directed_ridge', next_score)
    evaluate('rrf_local_directed', rrf(local_score, next_score, X))
    evaluate('global_plus_directed', global_score + PARAMS['beta'] * next_score)
    final_score = improved_score(train, global_score, local_score, next_score)
    evaluate('final_candidate', final_score)
    print(f'Срез {cut}: готово, прошло {time.time() - started:.0f} с')
    del X, B_local, B_next, global_score, local_score, next_score, final_score

validation = pd.DataFrame(results)
validation.to_csv('validation_results.csv', index=False)
comparison = validation.pivot(index='model', columns='cut', values='recall10')
comparison['mean'] = comparison.mean(axis=1)
display(comparison.reindex(MODEL_NAMES).rename(index=MODEL_NAMES))


Срез 1: готово, прошло 25 с


Срез 2: готово, прошло 51 с


Срез 3: готово, прошло 77 с


Срез 4: готово, прошло 103 с


Срез 5: готово, прошло 128 с


cut,1,2,3,4,5,mean
model,,,,,,
Популярное,0.03940,0.03626,0.03974,0.04371,0.03361,0.03854
EASE,0.08742,0.08477,0.09404,0.09702,0.08609,0.08987
EASE + соседи пропусков,0.08808,0.08841,0.09387,0.09950,0.09205,0.09238
EASE + недавняя локальная EASE,0.24470,0.24834,0.25182,0.25712,0.25414,0.25123
Directed ridge,0.24851,0.25497,0.26175,0.26142,0.26391,0.25811
RRF двух последовательных моделей,0.25381,0.26209,0.26523,0.26639,0.26689,0.26288
EASE + directed ridge,0.25298,0.26573,0.26821,0.27417,0.27152,0.26652
Финальный ансамбль,0.25728,0.26689,0.27732,0.27798,0.27483,0.27086


Командная модель дала 0,2447 на скрытом лидерборде. Новый ансамбль после отправки
дал **0,2480** по сообщению команды, то есть +0,0033 пункта recall@10.

На пяти временных срезах ранее получено 0,266523 у командного варианта и 0,270861
у итогового ансамбля. Эти срезы пересекаются и использовались при выборе параметров;
показатели не являются независимой оценкой. История гипотез и результаты отдельных
проверок описаны в `REPORT.md`.


## 7. Дополнительные проверки

Исторические подробные прогоны описаны в experiments.csv, если этот файл доступен.
В журнале указан протокол: temporal_oneout — временные срезы, random_10pct — случайное скрытие 10% истории.
Числа разных протоколов напрямую не сравниваем.

- **Бейзлайны и признаки.** На случайном сплите популярное дало 0.0692, популярное по демографии — 0.0741,
  по жанровому профилю — 0.0679. ItemKNN и iALS достигли 0.1495 и 0.1666, EASE — 0.1851.
- **Оценки как веса.** Бинарная EASE дала 0.1851, веса 1–5 — 0.1657. Удвоение высоких оценок
  и центрирование относительно среднего пользователя тоже не улучшили бинарный вариант.
- **Контекст и нормировки.** Проверены длина и сдвиги окон, затухание, вес предполагаемого хвоста,
  нормировка контекста, поправка на популярность, уменьшение веса активных пользователей и
  удаление отрицательных коэффициентов. Выигрыш отдельных настроек зависит от протокола.
- **Ранкер.** Проверены LightGBM с 17 признаками, последовательное добавление признаков
  и несколько разбиений. Его метрики в журнале относятся к случайному скрытию.
  Добавление признаков по одному не отделяет их собственный вклад от взаимодействий.
- **Последовательность.** Проверены двустороннее восстановление контекста, направленная ridge-модель,
  веса недавних событий и RRF. Финал выбираем по временной проверке из раздела 6.

Журнал содержит сохранённые результаты этих дополнительных прогонов.
Run All пересчитывает основные сравнения и финальный сабмит; весь поиск параметров повторно не запускается.

In [9]:
experiments_path = DATA / 'experiments.csv'
if experiments_path.is_file():
    experiments = pd.read_csv(experiments_path)
    assert set(experiments.protocol) <= {'random_10pct', 'temporal_oneout'}
    print(f'В подробном журнале {len(experiments)} измерений.')
    display(experiments.groupby(['protocol', 'family'], sort=False).size().rename('Прогонов').to_frame())
else:
    print('Исторический журнал experiments.csv отсутствует; запуск модели продолжается.')


Исторический журнал experiments.csv отсутствует; запуск модели продолжается.


## 8. Финальное обучение и сабмит

Обучаем все компоненты на доступных событиях, смешиваем скоры, исключаем уже оценённые
фильмы и выбираем 10 рекомендаций. Порядок пользователей берём из образца сабмита.
Результат — `submission_improved.csv` с колонками `user_id` и `item_id`; во второй
колонке 10 ID фильмов разделены пробелами. Этот вариант получил 0,2480 на лидерборде.


In [10]:
started = time.time()
X = to_matrix(events)

# Пять обученных матриц финального ансамбля; каждая сохраняется в отдельный снепшот (< 100 МБ)
TRAINERS = {
    'ease_global': lambda: ease_B(X, PARAMS['global_lam']),
    'ease_local': lambda: local_ease_B(events, PARAMS['window'], PARAMS['local_lam']),
    'ridge_tau3': lambda: directed_ridge(events, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon']),
    'ridge_tau3_recent': lambda: directed_ridge(events, PARAMS['tau'], PARAMS['ridge_lam'], PARAMS['horizon'],
                                                PARAMS['recent_strength'], PARAMS['recent_window']),
    'ridge_tau6': lambda: directed_ridge(events, PARAMS['tau_long'], PARAMS['ridge_lam'], PARAMS['horizon']),
}
M = {}
for name, train_fn in TRAINERS.items():
    path = MODELS / f'{name}.npz'
    if USE_SNAPSHOTS and path.exists():
        saved = np.load(path)
        assert saved['params'].item() == repr(PARAMS), f'{path} обучен с другими параметрами'
        M[name] = saved['B']
        print(f'{name}: загружена из {path}')
    else:
        M[name] = train_fn()
        MODELS.mkdir(exist_ok=True)
        np.savez_compressed(path, B=M[name], params=np.array(repr(PARAMS)))
        print(f'{name}: обучена и сохранена в {path}')

global_score = np.asarray(X @ M['ease_global'])
local_score = np.asarray(tail_weights(events, PARAMS['tau']) @ M['ease_local'])
next_score = np.asarray(tail_weights(events, PARAMS['tau'], PARAMS['horizon']) @ M['ridge_tau3'])
final_score = improved_score(events, global_score, local_score, next_score,
                             B_recent=M['ridge_tau3_recent'], B_long=M['ridge_tau6'])
recommendations = topk(final_score, X)

submission = pd.DataFrame({
    'user_id': np.arange(N_USERS),
    'item_id': [' '.join(map(str, row)) for row in recommendations],
}).set_index('user_id').loc[sample.user_id].reset_index()

seen = set(zip(events.user_id, events.item_id))
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.user_id.tolist() == sample.user_id.tolist()
assert submission.user_id.is_unique and len(submission) == N_USERS
assert not submission.isna().any().any()
assert recommendations.shape == (N_USERS, K)
assert all(len(set(row)) == K for row in recommendations)
assert recommendations.min() >= 0 and recommendations.max() < N_ITEMS
assert not any((u, int(i)) in seen for u, row in enumerate(recommendations) for i in row)

submission.to_csv('submission_improved.csv', index=False)
print(f'submission_improved.csv: {len(submission)} пользователей × {K} фильмов; {time.time() - started:.0f} с')
display(submission.head())


ease_global: загружена из models/ease_global.npz


ease_local: загружена из models/ease_local.npz


ridge_tau3: загружена из models/ridge_tau3.npz


ridge_tau3_recent: загружена из models/ridge_tau3_recent.npz


ridge_tau6: загружена из models/ridge_tau6.npz


submission_improved.csv: 6040 пользователей × 10 фильмов; 10 с


,user_id,item_id
0,0,3002 1332 1811 213 1527 1461 1044 2732 2687 3667
1,1,232 1246 3656 1039 2468 1868 1686 3153 208 3046
2,2,2354 3358 1956 438 3205 452 488 2342 1560 728
3,3,183 1322 3022 3656 3223 304 3390 899 2169 3441
4,4,1545 802 1337 2338 2653 1809 2222 2484 1287 488


## Итог

Финальная модель объединяет глобальную и локальную EASE с направленными ridge-моделями
на двух масштабах контекста. Для воспроизведения достаточно установить зависимости,
положить входные CSV в `data/` и выполнить его целиком. Результат сохраняется
в `submission_improved.csv`. Обученные матрицы сохраняются в `models/` и при повторном
запуске загружаются оттуда. Полный исторический журнал команды из 203 измерений недоступен.
